In [ ]:
#Necessary Dependencies 
import sys
sys.path.append("..")

import importlib
import tensorflow as tf
import DataModules.DataAugmentation
importlib.reload(DataModules.DataAugmentation)

from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Flatten, Dropout
from keras.applications.inception_v3 import InceptionV3
from keras.applications.inception_v3 import preprocess_input
from DataModules.DataAugmentation import get_datasets

In [ ]:
train_dataset1, test_dataset1, train_dataset2, test_dataset2, train_dataset3, test_dataset3, train_dataset4, test_dataset4, data_augmentation = get_datasets()

In [ ]:
#The Model for RNFl_DM

googlenet_rnfl_dm = InceptionV3(input_shape=IMAGE_SIZE + (3,), weights='imagenet', include_top=False)

for layer in googlenet_rnfl_dm.layers:
    layer.trainable = False

x1 = googlenet_rnfl_dm.output
x1 = data_augmentation()(x1)
x1 = GlobalAveragePooling2D()(x1)
x1 = Dense(10, activation='relu')(x1)
x1 = Dropout(.5)(x1)
x1 = Flatten()(x1)
googlenet_prediction1 = Dense(1, activation='sigmoid')(x1)

# create a model object
googlenet_model1 = Model(inputs=googlenet_rnfl_dm.inputs, outputs=googlenet_prediction1)

In [ ]:
#The Model for RNFL_TM

googlenet_rnfl_tm = InceptionV3(input_shape=IMAGE_SIZE + (3,), weights='imagenet', include_top=False)

for layer in googlenet_rnfl_tm.layers:
    layer.trainable = False

x2 = googlenet_rnfl_tm.output
x2 = data_augmentation()(x2)
x2 = GlobalAveragePooling2D()(x2)
x2 = Dense(10, activation='relu')(x2)
x2 = Dropout(.5)(x2)
x2 = Flatten()(x2)
googlenet_prediction2 = Dense(1, activation='sigmoid')(x2)

# create a model object
googlenet_model2 = Model(inputs=googlenet_rnfl_tm.inputs, outputs=googlenet_prediction2)

In [ ]:
#The Model for GCIPL_DM

googlenet_gcipl_dm = InceptionV3(input_shape=IMAGE_SIZE + (3,), weights='imagenet', include_top=False)

for layer in googlenet_gcipl_dm.layers:
    layer.trainable = False

x3 = googlenet_gcipl_dm.output
x3 = data_augmentation()(x3)
x3 = GlobalAveragePooling2D()(x3)
x3 = Dense(10, activation='relu')(x3)
x3 = Dropout(.5)(x3)
x3 = Flatten()(x3)
googlenet_prediction3 = Dense(1, activation='sigmoid')(x3)

# create a model object
googlenet_model3 = Model(inputs=googlenet_gcipl_dm.inputs, outputs=googlenet_prediction3)

In [ ]:
#The Model for GCIPL_TM

googlenet_gcipl_tm = InceptionV3(input_shape=IMAGE_SIZE + (3,), weights='imagenet', include_top=False)

for layer in googlenet_gcipl_tm.layers:
    layer.trainable = False

x4 = googlenet_gcipl_tm.output
x4 = data_augmentation()(x4)
x4 = GlobalAveragePooling2D()(x4)
x4 = Dense(10, activation='relu')(x4)
x4 = Dropout(.5)(x4)
x4 = Flatten()(x4)
googlenet_prediction4 = Dense(1, activation='sigmoid')(x4)

# create a model object
googlenet_model4 = Model(inputs=googlenet_gcipl_tm.inputs, outputs=googlenet_prediction4)

In [ ]:
histories = {}

models = {
    "GoogleNet_RNFL(DM)": (googlenet_model1, train_dataset1, test_dataset1),
    "GoogleNet_RNFL(TM)": (googlenet_model2, train_dataset2, test_dataset2),
    "GoogleNet_GCIPL(DM)": (googlenet_model3, train_dataset3, test_dataset3),
    "GoogleNet_GCIPL(TM)": (googlenet_model4, train_dataset4, test_dataset4)
}

for name, (model, train_dataset, test_dataset) in models.items():
    tf.keras.backend.clear_session()
    print(f"Training {name}...")

    model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
    )
    
    histories[name] = model.fit(
        train_dataset,
        validation_data=test_dataset,
        epochs=150
    )